In [0]:
from pyspark.sql.functions import *

df_series = spark.table("default.top_100_tmdb")
df_series.display()

In [0]:
df_series = df_series.filter(col('popularidade')>0).withColumn('base_fa', col("votos_tmdb")/col("popularidade")).orderBy(col('base_fa').desc())
df_series.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df_series = df_series.withColumn('rank', row_number().over(Window.orderBy(col('base_fa').desc())))
df_series = df_series.withColumn('tier', when(col('rank')<15, 'S').when(col('rank')<30, 'A').when(col('rank')<60, 'B')\
    .when(col('rank')<80, 'C').otherwise('D'))
df_series.display()


In [0]:
df_series = df_series.groupby("tier").agg(count('*').alias('contagem'))
df_series.display()